In [ ]:
# @title ⚙️ Install dependencies
!pip install requests plotly pandas numpy -q

In [ ]:
# @title 🔑 API Keys & Settings  { run: "auto" }

API_KEY  = "YOUR_API_KEY_HERE"   # ← paste your eToro API key
USER_KEY = "YOUR_USER_KEY_HERE"  # ← paste your eToro user key

# ── Chart settings ────────────────────────────────────────
TICKER        = "LLY"   # change to any eToro ticker
CANDLE_COUNT  = 150     # number of daily candles

# ── Indicator settings ────────────────────────────────────
EMA_FAST  = 7
EMA_MID   = 77
EMA_SLOW  = 231
BB_LENGTH = 20
BB_MULT   = 2.0
SAR_START = 0.02
SAR_INC   = 0.02
SAR_MAX   = 0.20

In [ ]:
# @title 📡 eToro API helpers
import uuid, requests, numpy as np, pandas as pd

BASE_URL = "https://public-api.etoro.com/api/v1"

def _headers():
    return {
        "x-api-key"    : API_KEY,
        "x-user-key"   : USER_KEY,
        "x-request-id" : str(uuid.uuid4()),
    }

def get_instrument_id(ticker: str):
    resp = requests.get(
        f"{BASE_URL}/market-data/search",
        headers=_headers(),
        params={"internalSymbolFull": ticker.upper(),
                "fields": "instrumentId,internalSymbolFull,displayName"}
    )
    resp.raise_for_status()
    data  = resp.json()
    items = data.get("items") or data.get("instruments") or data.get("data") or []
    if not items:
        raise ValueError(f"Ticker '{ticker}' not found on eToro.")
    item = items[0]
    iid  = item.get("instrumentId") or item.get("InstrumentID")
    name = item.get("displayName") or ticker
    return int(iid), str(name)

def fetch_candles(instrument_id: int, count: int) -> pd.DataFrame:
    url  = (f"{BASE_URL}/market-data/instruments/{instrument_id}"
            f"/history/candles/desc/OneDay/{count}")
    resp = requests.get(url, headers=_headers())
    resp.raise_for_status()
    raw     = resp.json()
    candles = raw["candles"][0]["candles"]
    rows = [{
        "date"  : pd.to_datetime(c["fromDate"]),
        "open"  : float(c["open"]),
        "high"  : float(c["high"]),
        "low"   : float(c["low"]),
        "close" : float(c["close"]),
    } for c in candles]
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    print(f"  Loaded {len(df)} candles  "
          f"({df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()})")
    return df

print("✓ API helpers ready")

In [ ]:
# @title 📐 Indicator functions

def calc_ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()

def calc_bb(close: pd.Series, length: int, mult: float):
    basis = close.ewm(span=length, adjust=False).mean()
    dev   = close.rolling(length).std()
    return basis, basis + mult * dev, basis - mult * dev

def calc_sar(df: pd.DataFrame,
             start: float = 0.02,
             inc:   float = 0.02,
             max_:  float = 0.20) -> pd.Series:
    """Parabolic SAR — mirrors Pine Script sar() function."""
    high, low = df["high"].values, df["low"].values
    n   = len(high)
    sar = np.full(n, np.nan)
    bull = True
    ep   = low[0]
    af   = start
    sar[0] = high[0]

    for i in range(1, n):
        ps = sar[i - 1]
        if bull:
            sar[i] = ps + af * (ep - ps)
            sar[i] = min(sar[i], low[i - 1])
            if i >= 2:
                sar[i] = min(sar[i], low[i - 2])
            if low[i] < sar[i]:
                bull, sar[i], ep, af = False, ep, low[i], start
            else:
                if high[i] > ep:
                    ep = high[i]
                    af = min(af + inc, max_)
        else:
            sar[i] = ps + af * (ep - ps)
            sar[i] = max(sar[i], high[i - 1])
            if i >= 2:
                sar[i] = max(sar[i], high[i - 2])
            if high[i] > sar[i]:
                bull, sar[i], ep, af = True, ep, high[i], start
            else:
                if low[i] < ep:
                    ep = low[i]
                    af = min(af + inc, max_)

    return pd.Series(sar, index=df.index)

print("✓ Indicators ready")

In [ ]:
# @title 📥 Fetch data from eToro
print(f"Resolving {TICKER}...")
instrument_id, name = get_instrument_id(TICKER)
print(f"  Found: {name}  (id={instrument_id})")

print(f"Fetching {CANDLE_COUNT} daily candles...")
df = fetch_candles(instrument_id, CANDLE_COUNT)

In [ ]:
# @title 📊 Build interactive chart
import plotly.graph_objects as go
from plotly.subplots import make_subplots

close  = df["close"]
dates  = df["date"]

ema7   = calc_ema(close, EMA_FAST)
ema77  = calc_ema(close, EMA_MID)
ema231 = calc_ema(close, EMA_SLOW)
basis, bb_up, bb_dn = calc_bb(close, BB_LENGTH, BB_MULT)
sar    = calc_sar(df, SAR_START, SAR_INC, SAR_MAX)

sar_bull = sar < close
sar_bear = ~sar_bull
exh_short = (df["open"] > bb_up) & (close > bb_up)
exh_long  = (df["open"] < bb_dn) & (close < bb_dn)

# ── theme ─────────────────────────────────────────────────
BG   = "#131722"
BG2  = "#1e222d"
GRID = "#2a2e39"
TEXT = "#d1d4dc"
MUTED = "#787b86"

fig = go.Figure()

# BB fill
fig.add_trace(go.Scatter(
    x=dates, y=bb_up, name="BB Upper", legendgroup="bb",
    line=dict(color="#43A047", width=1), showlegend=True,
))
fig.add_trace(go.Scatter(
    x=dates, y=bb_dn, name="BB Lower", legendgroup="bb",
    fill="tonexty", fillcolor="rgba(33,150,243,0.06)",
    line=dict(color="#FF6F00", width=1), showlegend=True,
))
fig.add_trace(go.Scatter(
    x=dates, y=basis, name="BB Basis (EMA20)", legendgroup="bb",
    line=dict(color="#1565C0", width=1.5, dash="dot"),
))

# Exhaustion highlight shapes
shapes = []
for i in df.index[exh_short]:
    shapes.append(dict(
        type="rect", xref="x", yref="paper",
        x0=dates[i], x1=dates[i], y0=0, y1=1,
        line=dict(color="rgba(239,83,80,0.5)", width=7),
        fillcolor="rgba(239,83,80,0.10)",
    ))
for i in df.index[exh_long]:
    shapes.append(dict(
        type="rect", xref="x", yref="paper",
        x0=dates[i], x1=dates[i], y0=0, y1=1,
        line=dict(color="rgba(38,166,154,0.5)", width=7),
        fillcolor="rgba(38,166,154,0.10)",
    ))

# Candlesticks
fig.add_trace(go.Candlestick(
    x=dates, open=df["open"], high=df["high"],
    low=df["low"], close=close, name="Price",
    increasing=dict(line=dict(color="#26a69a", width=1), fillcolor="#26a69a"),
    decreasing=dict(line=dict(color="#ef5350", width=1), fillcolor="#ef5350"),
))

# EMAs
fig.add_trace(go.Scatter(x=dates, y=ema7,
    name=f"EMA {EMA_FAST}", line=dict(color="#2fff00", width=2)))
fig.add_trace(go.Scatter(x=dates, y=ema77,
    name=f"EMA {EMA_MID}", line=dict(color="#ef5350", width=2)))
fig.add_trace(go.Scatter(x=dates, y=ema231,
    name=f"EMA {EMA_SLOW}", line=dict(color="#ffffff", width=1.5)))

# SAR dots
fig.add_trace(go.Scatter(
    x=dates[sar_bull], y=sar[sar_bull], name="SAR bullish",
    mode="markers", marker=dict(color="#00e676", size=5),
))
fig.add_trace(go.Scatter(
    x=dates[sar_bear], y=sar[sar_bear], name="SAR bearish",
    mode="markers", marker=dict(color="#ef5350", size=5),
))

# ── layout ────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=f"<b>{name}</b> ({TICKER})  ·  BB + EMA + Parabolic SAR  ·  Daily",
        font=dict(color=TEXT, size=14, family="monospace"), x=0.01,
    ),
    paper_bgcolor=BG, plot_bgcolor=BG2,
    font=dict(color=TEXT, family="monospace"),
    height=650,
    xaxis=dict(
        showgrid=True, gridcolor=GRID, gridwidth=0.5,
        tickfont=dict(color=MUTED, size=10),
        rangeslider=dict(visible=False),
        showspikes=True, spikecolor="#aaaaaa",
        spikedash="dot", spikethickness=1, spikemode="across",
        tickformat="%b %d, %Y",
        rangeselector=dict(
            bgcolor=BG2, activecolor=GRID,
            bordercolor=GRID, borderwidth=1,
            font=dict(color=TEXT, size=10, family="monospace"),
            buttons=[
                dict(count=1,  label="1M",  step="month", stepmode="backward"),
                dict(count=3,  label="3M",  step="month", stepmode="backward"),
                dict(count=6,  label="6M",  step="month", stepmode="backward"),
                dict(step="all", label="All"),
            ],
        ),
    ),
    yaxis=dict(
        showgrid=True, gridcolor=GRID, gridwidth=0.5,
        tickfont=dict(color=MUTED, size=10), side="right",
        showspikes=True, spikecolor="#aaaaaa",
        spikedash="dot", spikethickness=1, spikemode="across",
    ),
    hovermode="x unified",
    hoverlabel=dict(
        bgcolor=BG2, bordercolor=GRID,
        font=dict(color=TEXT, size=11, family="monospace"),
    ),
    legend=dict(
        bgcolor="rgba(19,23,34,0.85)", bordercolor=GRID, borderwidth=1,
        font=dict(color=TEXT, size=11, family="monospace"),
        orientation="v", x=0.01, y=0.99,
        xanchor="left", yanchor="top",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
    shapes=shapes,
    margin=dict(l=10, r=60, t=50, b=40),
)

fig.show()
print(f"\n  Exhaustion bars highlighted:")
print(f"    Short (full candle above upper BB): {exh_short.sum()}")
print(f"    Long  (full candle below lower BB): {exh_long.sum()}")

In [ ]:
# @title 📉 Optional — Add RSI panel below chart
from plotly.subplots import make_subplots

def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / loss.replace(0, 1e-10)
    return 100 - (100 / (1 + rs))

rsi = calc_rsi(close)

fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

# ── copy all main chart traces to row 1 ──
for trace in fig.data:
    fig2.add_trace(trace, row=1, col=1)

# ── RSI ──
fig2.add_trace(go.Scatter(
    x=dates, y=rsi, name="RSI 14",
    line=dict(color="#BA7517", width=1.5),
), row=2, col=1)

# Overbought / oversold lines
for level, color in [(70, "rgba(239,83,80,0.4)"), (30, "rgba(38,166,154,0.4)")]:
    fig2.add_hline(y=level, line=dict(color=color, width=1, dash="dash"),
                   row=2, col=1)

fig2.add_hrect(y0=70, y1=100,
               fillcolor="rgba(239,83,80,0.05)", line_width=0,
               row=2, col=1)
fig2.add_hrect(y0=0,  y1=30,
               fillcolor="rgba(38,166,154,0.05)", line_width=0,
               row=2, col=1)

fig2.update_layout(
    **{k: v for k, v in fig.layout.to_plotly_json().items()
       if k not in ("shapes",)},
    height=800,
    shapes=shapes,
    title=dict(
        text=f"<b>{name}</b> ({TICKER})  ·  BB + EMA + SAR + RSI  ·  Daily",
        font=dict(color=TEXT, size=14, family="monospace"), x=0.01,
    ),
)
fig2.update_yaxes(
    showgrid=True, gridcolor=GRID, tickfont=dict(color=MUTED, size=9),
    showspikes=True, spikecolor="#aaaaaa", spikedash="dot",
    spikethickness=1, spikemode="across",
    row=2, col=1,
)
fig2.update_xaxes(showspikes=True, spikecolor="#aaaaaa",
                  spikedash="dot", spikethickness=1, spikemode="across")

fig2.show()